# GPT-OSS Reference Profiling Notebook

Profile GPT-OSS with dummy BF16 weights and the forward path in `model.py`.

## Setup

1. Run `uv sync`.
2. Install a CUDA-compatible PyTorch build on the target machine.
3. Start Jupyter with `uv run jupyter lab`.
4. Open this notebook.

## Decode Semantics

Strict-reference mode uses one prompt forward pass for prefill. If `generated_tokens > 0`, the first new token is sampled from the prefill logits. `decode_total` measures only the follow-on full-sequence passes for the remaining generated tokens.


In [1]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (CWD, *CWD.parents) if (path / "pyproject.toml").exists()),
    CWD,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from profiling import (
    DEFAULT_GENERATED_SWEEP,
    DEFAULT_PREFILL_SWEEP,
    TimingConfig,
    WorkloadConfig,
    build_reference_model,
    format_bytes,
    model_config_from_preset,
    plot_level0_breakdown,
    plot_level1_breakdown,
    plot_level2_heatmap,
    plot_workload_grid,
    preflight_report,
    run_workload,
    run_workload_sweep,
    summarize_results,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
PRESET_NAME = "gpt-oss-20b"
MODEL_OVERRIDES = {
    # "num_hidden_layers": 12,
    # "num_experts": 16,
    # "num_attention_heads": 32,
}

DEVICE = "cuda"
SEED = 0

SINGLE_WORKLOAD = {
    "prefill_tokens": 2048,
    "generated_tokens": 16,
}

SWEEP_PREFILL = list(DEFAULT_PREFILL_SWEEP)
SWEEP_GENERATED = list(DEFAULT_GENERATED_SWEEP)

TIMING = TimingConfig(
    warmup_iters=2,
    measure_iters=5,
    seed=SEED,
    enable_torch_profiler=False,
    trace_output_dir="traces",
)


In [3]:
arch_cfg = model_config_from_preset(PRESET_NAME, MODEL_OVERRIDES)
single_workload = WorkloadConfig(**SINGLE_WORKLOAD)
report = preflight_report(arch_cfg, single_workload, device=DEVICE)

arch_table = pd.DataFrame([
    {"field": key, "value": value}
    for key, value in asdict(arch_cfg).items()
])

preflight_table = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in report.items()
])
preflight_table["pretty_value"] = preflight_table.apply(
    lambda row: format_bytes(row["value"])
    if isinstance(row["value"], int) and ("bytes" in row["metric"])
    else row["value"],
    axis=1,
)

display(arch_table)
display(preflight_table[["metric", "pretty_value"]])


,field,value
0,num_hidden_layers,24.0
1,num_experts,32.0
2,experts_per_token,4.0
3,vocab_size,201088.0
4,hidden_size,2880.0
5,intermediate_size,2880.0
6,swiglu_limit,7.0
7,head_dim,64.0
8,num_attention_heads,64.0
9,num_key_value_heads,8.0


,metric,pretty_value
0,embedding_params,579133440
1,transformer_block_attention_params,26553024
2,transformer_block_mlp_params,796633952
3,transformer_block_total_params,823186976
4,final_norm_params,2880
5,unembedding_params,579133440
6,total_params,20914757184
7,bf16_param_count,20914616064
8,fp32_param_count,141120
9,dense_weight_bytes,38.96 GiB


In [ ]:
model = build_reference_model(arch_cfg, device=DEVICE, seed=SEED)
single_rows = run_workload(
    arch_cfg,
    single_workload,
    TIMING,
    device=DEVICE,
    preset_name=PRESET_NAME,
    model=model,
)
single_summary = summarize_results(single_rows)

trace_path = single_rows.attrs.get("trace_path")
if trace_path:
    print(f"Chrome trace written to: {trace_path}")


In [ ]:
display(single_summary["level0_metrics"])
display(single_summary["level1"].sort_values(["phase", "scope_name"]))
display(
    single_summary["level2"]
    .sort_values(["phase", "component", "layer_idx", "scope_name"])
    .reset_index(drop=True)
)


In [ ]:
plot_level0_breakdown(single_summary["level0_metrics"])
plot_level1_breakdown(single_summary["level1"], phase="combined")
plot_level2_heatmap(
    single_summary["level2"],
    phase="prefill",
    component="attention",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)
plot_level2_heatmap(
    single_summary["level2"],
    phase="prefill",
    component="mlp",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)


In [ ]:
sweep_rows = run_workload_sweep(
    arch_cfg,
    prefill_tokens=SWEEP_PREFILL,
    generated_tokens=SWEEP_GENERATED,
    timing_cfg=TIMING,
    device=DEVICE,
    preset_name=PRESET_NAME,
)
sweep_summary = summarize_results(sweep_rows)

display(
    sweep_summary["level0_metrics"]
    .sort_values(["prefill_tokens", "generated_tokens"])
    .reset_index(drop=True)
)

plot_workload_grid(sweep_summary["level0_metrics"], value="workload_total")
plot_workload_grid(sweep_summary["level0_metrics"], value="avg_ms_per_new_token")
plot_level1_breakdown(sweep_summary["level1"], phase="combined")
